In [9]:
# Importações

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import ee

print("OK")

OK


In [3]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
ee.Authenticate()
ee.Initialize(project="spatial-yew-490017-r3")
print(ee.String("Hello from the Earth Engine servers!").getInfo())

Hello from the Earth Engine servers!


In [35]:
# Foco de calor INPE (CSV)

df_inpe = pd.read_csv('files/inpe/focos_br_todos-sats_2025.csv')

print("Shape:", df_inpe.shape)
print("\nColunas:", df_inpe.columns.tolist())
print("\nTipos de dado:\n", df_inpe.dtypes)
print("\nPrimeiras linhas:\n", df_inpe.head())


Shape: (3466399, 13)

Colunas: ['latitude', 'longitude', 'data_pas', 'satelite', 'pais', 'estado', 'municipio', 'bioma', 'numero_dias_sem_chuva', 'precipitacao', 'risco_fogo', 'id_area_industrial', 'frp']

Tipos de dado:
 latitude                 float64
longitude                float64
data_pas                     str
satelite                     str
pais                         str
estado                       str
municipio                    str
bioma                        str
numero_dias_sem_chuva    float64
precipitacao             float64
risco_fogo               float64
id_area_industrial         int64
frp                      float64
dtype: object

Primeiras linhas:
    latitude  longitude             data_pas satelite    pais  \
0  -19.7334 -55.399899  2025-01-01 00:35:00  METOP-B  Brasil   
1  -15.2145 -60.173901  2025-01-01 00:38:00  METOP-B  Brasil   
2  -15.2090 -60.198799  2025-01-01 00:38:00  METOP-B  Brasil   
3  -12.2099 -49.326199  2025-01-01 00:38:00  METOP-B  Brasi

,latitude,longitude,data_pas,satelite,pais,estado,municipio,bioma,numero_dias_sem_chuva,precipitacao,risco_fogo,id_area_industrial,frp
0,-19.7334,-55.399899,2025-01-01 00:35:00,METOP-B,Brasil,MATO GROSSO DO SUL,AQUIDAUANA,Pantanal,7.0,0.00,0.74,0,NaN
1,-15.2145,-60.173901,2025-01-01 00:38:00,METOP-B,Brasil,MATO GROSSO,VILA BELA DA SANTÍSSIMA TRINDADE,Amazônia,4.0,0.72,0.65,0,NaN
2,-15.2090,-60.198799,2025-01-01 00:38:00,METOP-B,Brasil,MATO GROSSO,VILA BELA DA SANTÍSSIMA TRINDADE,Amazônia,3.0,1.32,0.58,0,NaN
3,-12.2099,-49.326199,2025-01-01 00:38:00,METOP-B,Brasil,TOCANTINS,FIGUEIRÓPOLIS,Cerrado,1.0,0.00,0.10,0,NaN
4,-12.2082,-49.318901,2025-01-01 00:38:00,METOP-B,Brasil,TOCANTINS,FIGUEIRÓPOLIS,Cerrado,1.0,0.00,0.14,0,NaN


In [18]:
# GABAM (Earth Engine)

gabam = ee.ImageCollection("projects/sat-io/open-datasets/GABAM")

print("Tipo:", type(gabam))
print("Total de imagens:", gabam.size().getInfo())

first_gabam = gabam.first()
print("\nBandas:", first_gabam.bandNames().getInfo())
print("\nPropriedades:", first_gabam.propertyNames().getInfo())
print("\nProjeção:", first_gabam.projection().getInfo())
print("\nEscala nominal (m):", first_gabam.projection().nominalScale().getInfo())

datas = gabam.aggregate_array('system:time_start').getInfo()
anos = sorted(set(pd.to_datetime(d, unit='ms').year for d in datas if d is not None))
print("\nAnos disponíveis:", anos)

Tipo: <class 'ee.imagecollection.ImageCollection'>
Total de imagens: 14614

Bandas: ['b1']

Propriedades: ['system:time_start', 'num_bands', 'id_no', 'ysize', 'system:footprint', 'system:time_end', 'system:version', 'xsize', 'system:id', 'system:asset_size', 'system:index', 'system:bands', 'system:band_names']

Projeção: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.00025000000000000017, 0, 1.1368683772161603e-13, 0, -0.00025000000000000017, -5.684341886080802e-14]}

Escala nominal (m): 27.82987269831841

Anos disponíveis: [1985, 1987, 1989, 1992, 1995, 1996, 1998, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


In [19]:
#  MODIS MCD64A1 (Earth Engine)

modis = ee.ImageCollection("MODIS/061/MCD64A1")

print("Total de imagens na coleção:", modis.size().getInfo())

first_modis = modis.first()
print("\nBandas:", first_modis.bandNames().getInfo())
print("\nPropriedades:", first_modis.propertyNames().getInfo())
print("\nData da primeira imagem:", ee.Date(first_modis.get('system:time_start')).format().getInfo())

last_modis = modis.sort('system:time_start', False).first()
print("Data da última imagem:", ee.Date(last_modis.get('system:time_start')).format().getInfo())

print("\nEscala nominal (m):", first_modis.projection().nominalScale().getInfo())

Total de imagens na coleção: 307

Bandas: ['BurnDate', 'Uncertainty', 'QA', 'FirstDay', 'LastDay']

Propriedades: ['system:time_start', 'google:max_source_file_timestamp', 'num_tiles', 'system:footprint', 'system:time_end', 'system:version', 'system:id', 'system:asset_size', 'system:index', 'system:bands', 'system:band_names']

Data da primeira imagem: 2000-11-01T00:00:00
Data da última imagem: 2026-05-01T00:00:00

Escala nominal (m): 463.31271652791656


In [25]:
# AAF ICMBio (Shapefile)

gdf_aaf = gpd.read_file('files/aaf_2025/aaf_2025.shp')

print("Shape:", gdf_aaf.shape)
print("\nColunas:", gdf_aaf.columns.tolist())
print("\nTipos de dado:\n", gdf_aaf.dtypes)

Shape: (3133, 25)

Colunas: ['cnuc', 'nome_uc', 'area_ha', 'acao', 'tipo', 'satelite', 'obs', 'juliano', 'data', 'ano', 'mes_nome', 'mes_num', 'categoria', 'local', 'area_uc', 'area_ent', 'bioma', 'gr_nome', 'ngi', 'cr', 'GlobalID', 'Shape__Are', 'Shape__Len', 'ct', 'geometry']

Tipos de dado:
 cnuc                     str
nome_uc                  str
area_ha              float64
acao                     str
tipo                     str
satelite                 str
obs                      str
juliano              float64
data          datetime64[ms]
ano                    int64
mes_nome                 str
mes_num                  str
categoria                str
local                    str
area_uc              float64
area_ent             float64
bioma                    str
gr_nome                  str
ngi                      str
cr                       str
GlobalID              object
Shape__Are           float64
Shape__Len           float64
ct                       str
geometry

In [31]:
mbfogo = ee.FeatureCollection(
    "projects/mapbiomas-public/assets/brazil/fire/collection5/"
    "mapbiomas_fire_collection5_annual_burned_vectors/mbfogo_col5_2025_v1"
)

print("Total de feições:", mbfogo.size().getInfo())

first_feature = mbfogo.first()
print("\nPropriedades disponíveis:", first_feature.propertyNames().getInfo())
print("\nInfo completa da primeira feição:", first_feature.getInfo())

Total de feições: 306190

Propriedades disponíveis: ['DN', 'system:index', 'id']

Info completa da primeira feição: {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-43.052299650798716, -3.916027128676686], [-43.052299650798716, -3.9162946738228794], [-43.05202761596238, -3.916294654162223], [-43.05202761596238, -3.9165667121155843], [-43.051760098800315, -3.916566698549886], [-43.051760098800315, -3.9168342495028403], [-43.05068097758389, -3.9168342463033774], [-43.050680977583895, -3.9165666911593804], [-43.05014140836594, -3.916566715921887], [-43.05014140836594, -3.9162947090908933], [-43.04852718569507, -3.9162946617378647], [-43.04852718569508, -3.9157551510815707], [-43.04825520471477, -3.9157551226286387], [-43.04825520471477, -3.9154875654862], [-43.04798766765605, -3.9154876043879816], [-43.04798766765605, -3.914948042129149], [-43.04771568378309, -3.9149480102121275], [-43.04771568378309, -3.914680515097297], [-43.047448143879, -3.9146804920196567], [-43